In [ ]:
# ============================================================
# 045_Notebook_builder.ipynb
# ============================================================
#
# Overview
# ----------------
# Build and repair target notebooks by consuming real tasks from TASKS_DB and
# executing a deterministic patch-and-run loop that writes evidence back to Notion.
#
# This notebook is the “control plane” that:
#   1) Reads the next actionable Task from TASKS_DB (no synthetic smoke tasks)
#   2) Resolves Notion data_source_id once and builds repo accessors (SPEC-aligned)
#   3) Prepares a sandbox copy of the target notebook (never edits this notebook)
#   4) Enqueues deterministic steps into StateStore (scaffold/patch/verify)
#   5) Executes the loop (run_one_step) until the queue is idle
#
# The loop is designed to be:
#   - Spec-aligned (Notion queries use data_sources only; database_id is never queried directly)
#   - Evidence-driven (actions are decided from execution results, not assumptions)
#   - Safe (sandbox-only patching + backups + artifacted diffs + rollback)
#
# Inputs / Outputs
# ----------------
# Inputs:
#   - notebooks/env.txt
#       - NOTION_TOKEN
#       - NOTION_TASKS_DB_ID
#       - NOTION_PROPOSALS_DB_ID
#       - NOTION_RUNS_DB_ID
#       - NOTION_DECISIONS_DB_ID (optional: only if you implement Decision writing later)
#   - TASKS_DB content (real tasks)
#       - Entry Point: target notebook path (relative to project-root)
#       - Objective / Acceptance Criteria: outcome + completion definition
#       - Run Policy: run mode defaults, timeouts, quality gates policy (if used)
#
# Outputs:
#   - Notion audit trail (current scope):
#       - PROPOSALS_DB: ChangeSets created/updated by the orchestrator (if enabled in one_loop)
#       - RUNS_DB: execution evidence for sandbox notebook runs (if enabled in one_loop)
#       - TASKS_DB: (optional) status updates if you wire it in this notebook
#   - Local artifacts:
#       - artifacts/<run_id>/... (rendered notebooks, logs, traces, diffs)
#       - state/backups/ (rollback snapshots)
#       - state/state.json (queue + last_error + progress markers)
#
# Structure
# ----------------
# Cell 01: Move CWD to project root and set import path (robust path discovery)
# Cell 02: Load notebooks/env.txt (dotenv) and validate required env vars
# Cell 03: Build Notion client + resolve data_source_id ONCE + build repos (SPEC-aligned caching)
#
# Cell 04: Fetch “next Task to work on” from TASKS_DB (data_sources query)
#          - Filter: Status in (READY, RUNNING) and Priority order
#          - Select a single task deterministically (highest priority, oldest first)
#          - Read Task fields: Entry Point / Objective / AC / Run Policy
#
# Cell 05: Derive execution config from Task
#          - target_notebook_path (from Entry Point)
#          - sandbox_notebook_path (= <target>__sandbox.ipynb)
#          - run policy (prefix/full strategy, timeouts, optional quality gates)
#
# Cell 06: Prepare sandbox + initialize machine state (StateStore)
#          - create/refresh sandbox copy
#          - ensure state/state.json exists
#          - optional: clear queue for this Task run
#
# Cell 07: Create or resume a working Proposal chain for this Task (if used)
#          - If an active Proposal exists, resume it
#          - Else create a new Proposal (DRAFT) tied to the Task
#
# Cell 08: Bootstrap enqueue (deterministic)
#          - Enqueue minimal steps needed to start:
#              - SCAFFOLD_HEADERS (Cell00/01/02 + per-cell headers as needed)
#              - VERIFY_NOTEBOOK (PREFIX up to Cell02)
#          - (Optional) If you have a planner step:
#              - enqueue LLM_PLAN / APPLY_PATCH for the next target cell
#
# Cell 09: Execute the loop (one_step / until idle)
#          - run_one_step() repeatedly
#          - after each step:
#              - artifacts are written under artifacts/<run_id>/
#              - state.json is updated (last_error / progress / debounce digest)
#
# Notes
# ----------------
# - This notebook must never patch or execute itself. It only controls other notebooks.
# - All Notion queries must use POST /v1/data_sources/{data_source_id}/query (SPEC constraint).
# - Resolve data_source_id once (Cell 03), cache RESOLVED_DB, and never deep-scan again.
# - Use sandbox notebooks for all patches/execution; keep originals immutable unless explicitly promoted later.
# - If you later add Decision handling (DECISIONS_DB), implement it as an explicit step AFTER Cell09.


In [2]:
# ============================================================
# Cell 01 — Resolve project root and configure import path
# ============================================================
# Overview:
#   Ensure the notebook is executed with project-root as the current working
#   directory (CWD), and that Python can import modules from src/.
#
#   This cell performs robust path discovery:
#     - Starts from the current working directory
#     - Walks upward until a directory containing "src/" is found
#     - Treats that directory as project-root
#     - Forces os.chdir(project-root)
#     - Inserts project-root at the front of sys.path
#
#   This guarantees that:
#     - Relative paths like "notebooks/...", "state/...", "outputs/..." are stable
#     - Imports such as "from src.xxx import yyy" work consistently
#
# Inputs / Outputs:
#   Inputs:
#     - Current working directory (where Jupyter was launched)
#   заставляет:
#     - os.getcwd() == project-root
#     - sys.path[0] == project-root
#
# Notes:
#   - This notebook (044) is a control-plane notebook and must never be patched
#     or executed by the build loop itself.
#   - All subsequent cells assume CWD == project-root.
# ============================================================

from pathlib import Path
import os
import sys

# --- Discover project root (directory containing "src/") ---
cwd = Path.cwd()
project_root = None

cursor = cwd
for _ in range(8):  # safety limit to avoid infinite loop
    if (cursor / "src").is_dir():
        project_root = cursor
        break
    if cursor.parent == cursor:
        break
    cursor = cursor.parent

if project_root is None:
    raise RuntimeError(
        "Project root not found. "
        "Expected a directory containing 'src/'. "
        "Please launch Jupyter from the project root."
    )

# --- Enforce project-root as CWD ---
os.chdir(project_root)

# --- Ensure project-root is first on sys.path ---
root_str = str(project_root.resolve())
if sys.path[0] != root_str:
    if root_str in sys.path:
        sys.path.remove(root_str)
    sys.path.insert(0, root_str)

# --- Sanity checks ---
print("Project root resolved to:", project_root)
print("CWD:", Path.cwd())
print("sys.path[0]:", sys.path[0])
print("src/ exists:", (project_root / "src").exists())
print("notebooks/ exists:", (project_root / "notebooks").exists())


Project root resolved to: /Users/yuetoya/projects/researchOS100-private
CWD: /Users/yuetoya/projects/researchOS100-private
sys.path[0]: /Users/yuetoya/projects/researchOS100-private
src/ exists: True
notebooks/ exists: True


In [3]:
# ============================================================
# Cell 02 — Load notebooks/env.txt and validate environment variables
# ============================================================
# Overview:
#   Load environment variables from notebooks/env.txt using python-dotenv
#   and validate that all required variables for Notion integration exist.
#
#   This cell establishes a single, explicit source of truth for configuration:
#     - No implicit OS-level env assumptions
#     - No fallback to other .env files
#     - All downstream cells depend on these variables being present
#
# Inputs / Outputs:
#   Inputs:
#     - notebooks/env.txt
#         - NOTION_TOKEN
#         - NOTION_TASKS_DB_ID
#         - NOTION_PROPOSALS_DB_ID
#         - NOTION_RUNS_DB_ID
#         - NOTION_DECISIONS_DB_ID
#   Outputs:
#     - Environment variables loaded into os.environ
#     - Local Python variables bound for convenience
#
# Notes:
#   - env.txt must live under notebooks/ (not project root).
#   - All IDs are database_id (UUID), not data_source_id.
#   - data_source_id resolution happens later (Cell 03) and only once.
# ============================================================

import os
from pathlib import Path
from dotenv import load_dotenv

# --- Locate notebooks/env.txt explicitly ---
NOTEBOOKS_DIR = Path.cwd() / "notebooks"
ENV_PATH = NOTEBOOKS_DIR / "env.txt"

if not ENV_PATH.exists():
    raise FileNotFoundError(
        f"env.txt not found at expected location: {ENV_PATH}\n"
        "Please place env.txt under the notebooks/ directory."
    )

# --- Load environment variables ---
load_dotenv(dotenv_path=ENV_PATH)

# --- Required environment variables (SPEC-aligned) ---
NOTION_TOKEN = os.getenv("NOTION_TOKEN")
NOTION_TASKS_DB_ID = os.getenv("NOTION_TASKS_DB_ID")
NOTION_PROPOSALS_DB_ID = os.getenv("NOTION_PROPOSALS_DB_ID")
NOTION_RUNS_DB_ID = os.getenv("NOTION_RUNS_DB_ID")
NOTION_DECISIONS_DB_ID = os.getenv("NOTION_DECISIONS_DB_ID")

required_env = {
    "NOTION_TOKEN": NOTION_TOKEN,
    "NOTION_TASKS_DB_ID": NOTION_TASKS_DB_ID,
    "NOTION_PROPOSALS_DB_ID": NOTION_PROPOSALS_DB_ID,
    "NOTION_RUNS_DB_ID": NOTION_RUNS_DB_ID,
    "NOTION_DECISIONS_DB_ID": NOTION_DECISIONS_DB_ID,
}

missing = [key for key, value in required_env.items() if not value]

if missing:
    raise RuntimeError(
        "Missing required environment variables in notebooks/env.txt:\n"
        + "\n".join(f"  - {k}" for k in missing)
    )

# --- Sanity output (safe, no secrets) ---
print("Environment variables loaded successfully:")
for k in required_env.keys():
    print(f"  - {k}: [OK]")


Environment variables loaded successfully:
  - NOTION_TOKEN: [OK]
  - NOTION_TASKS_DB_ID: [OK]
  - NOTION_PROPOSALS_DB_ID: [OK]
  - NOTION_RUNS_DB_ID: [OK]
  - NOTION_DECISIONS_DB_ID: [OK]


In [4]:
# ============================================================
# Cell 03 — Build Notion client, resolve data_source_id ONCE, and build repos
# ============================================================
# Overview:
#   Initialize the Notion client and resolve data_source_id for each required
#   database exactly once (SPEC-aligned). Cache the resolved mapping in-memory
#   as RESOLVED_DB and build repository wrappers (repos) for all subsequent
#   reads/writes.
#
#   SPEC constraints (must hold):
#     - Never call POST /v1/databases/{database_id}/query
#     - Use GET /v1/databases/{database_id} for metadata/schema
#     - Query content ONLY via POST /v1/data_sources/{data_source_id}/query
#     - Resolve data_source_id by:
#         1) GET database object
#         2) recursively scan JSON for UUID-like strings
#         3) validate candidates by POST data_sources/{candidate}/query {"page_size": 1}
#         4) select first candidate that succeeds
#     - Deep scanning must happen only in this cell, once per DB
#
# Inputs / Outputs:
#   Inputs:
#     - Environment variables from Cell 02:
#         NOTION_TOKEN
#         NOTION_TASKS_DB_ID / NOTION_PROPOSALS_DB_ID / NOTION_RUNS_DB_ID / NOTION_DECISIONS_DB_ID
#   Outputs:
#     - client: Notion client instance
#     - RESOLVED_DB: dict mapping DB logical name -> {database_id, data_source_id}
#     - repos: NotionRepos wrapper providing typed accessors
#
# Notes:
#   - Do not re-run deep scanning in later cells. Only use cached RESOLVED_DB.
#   - IDs may appear as 32-hex or hyphenated UUID; normalization is handled by resolver.
# ============================================================

from src.notion.client import build_notion_client_from_env, NotionDataSourceResolver
from src.notion.repos import ResolvedDBRegistry, build_repos

# --- Build Notion client (reads NOTION_TOKEN from environment) ---
client = build_notion_client_from_env()

# --- Resolve data_source_id ONCE (SPEC-aligned) ---
resolver = NotionDataSourceResolver(client)


RESOLVED_DB = {}
RESOLVED_DB["TASKS_DB"] = resolver.resolve_once(name="TASKS_DB", database_id=NOTION_TASKS_DB_ID).__dict__
RESOLVED_DB["PROPOSALS_DB"] = resolver.resolve_once(name="PROPOSALS_DB", database_id=NOTION_PROPOSALS_DB_ID).__dict__
RESOLVED_DB["RUNS_DB"] = resolver.resolve_once(name="RUNS_DB", database_id=NOTION_RUNS_DB_ID).__dict__
RESOLVED_DB["DECISIONS_DB"] = resolver.resolve_once(name="DECISIONS_DB", database_id=NOTION_DECISIONS_DB_ID).__dict__

wtu_db_id = os.environ.get("NOTION_WEEKLY_TARGET_UPDATE_DB_ID")
if wtu_db_id:
    RESOLVED_DB["WEEKLY_TARGET_UPDATE_DB"] = resolver.resolve_once(
        name="WEEKLY_TARGET_UPDATE_DB",
        database_id=wtu_db_id,
    ).__dict__



print("RESOLVED_DB (cached for this notebook run):")
for k, v in RESOLVED_DB.items():
    print(f"  - {k}: database_id={v['database_id']}  data_source_id={v['data_source_id']}")

# --- Build repos (all future Notion operations go through this) ---
repos = build_repos(
    notion_client=client,
    resolved_registry=ResolvedDBRegistry(RESOLVED_DB),
)


import os
from src.llm.claude_client import ClaudeClient

# 必要なら環境変数で設定（あなたの実装に合わせて）
# os.environ["ANTHROPIC_API_KEY"] = "..."  # すでに設定済みなら不要

claude = ClaudeClient()
print("[info] claude client ready ✅")


RESOLVED_DB (cached for this notebook run):
  - TASKS_DB: database_id=3028e0e4-d162-80e1-aad7-fd8bf9a5be2e  data_source_id=3028e0e4-d162-8036-82e6-000b28a36c25
  - PROPOSALS_DB: database_id=3028e0e4-d162-80f5-8b23-e0f2af6f8820  data_source_id=3028e0e4-d162-8034-ad55-000b55236706
  - RUNS_DB: database_id=3028e0e4-d162-80bf-b757-eba87c02e0fb  data_source_id=3028e0e4-d162-80f1-b87c-000bbe4f56bb
  - DECISIONS_DB: database_id=3028e0e4-d162-8025-835a-caa9529345f3  data_source_id=3028e0e4-d162-80af-8c7e-000b85b29918
  - WEEKLY_TARGET_UPDATE_DB: database_id=2ff8e0e4-d162-80c0-b934-c9639ca5069a  data_source_id=2ff8e0e4-d162-8041-a4a4-000bcc5e745e
[info] claude client ready ✅


In [5]:
# ============================================================
# Cell 04 — Select a Task from TASKS_DB (ipywidgets UI)
# ============================================================
# Overview:
#   Present actionable Tasks (Status in READY/RUNNING) as a dropdown list and let
#   the user select exactly one Task to run. After selection, this cell outputs:
#     - task_page
#     - TASK_PAGE_ID
#     - task_fields
#
# Inputs / Outputs:
#   Inputs:
#     - repos (from Cell 03)
#   Outputs:
#     - task_page (dict)
#     - TASK_PAGE_ID (str)
#     - task_fields (dict)
#
# Notes:
#   - Requires ipywidgets. If not installed: pip install ipywidgets
#   - This is a human-in-the-loop selector (recommended in early phase).
# ============================================================

from typing import Optional, Dict, Any, List, Tuple
from datetime import datetime

import ipywidgets as widgets
from IPython.display import display, clear_output

# -------------------------
# Helpers
# -------------------------

PRIORITY_ORDER = {"P0": 0, "P1": 1, "P2": 2, "P3": 3}

def _select_name(prop: Dict[str, Any]) -> Optional[str]:
    if not prop or prop.get("type") != "select":
        return None
    v = prop.get("select")
    return v.get("name") if v else None

def _rich_text(prop: Dict[str, Any]) -> str:
    if not prop:
        return ""
    t = prop.get("type")
    if t == "rich_text":
        parts = prop.get("rich_text") or []
        return "".join(p.get("plain_text", "") for p in parts)
    if t == "title":
        parts = prop.get("title") or []
        return "".join(p.get("plain_text", "") for p in parts)
    return ""

def _extract_task_fields(page: Dict[str, Any]) -> Dict[str, Any]:
    props = page.get("properties") or {}
    return {
        "title": _rich_text(props.get("Title")),
        "status": _select_name(props.get("Status")),
        "priority": _select_name(props.get("Priority")),
        "domain": _select_name(props.get("Domain")),
        "entry_point": _rich_text(props.get("Entry Point")),
        "objective": _rich_text(props.get("Objective")),
        "acceptance_criteria": _rich_text(props.get("Acceptance Criteria")),
        "constraints": _rich_text(props.get("Constraints")),
        "run_policy": _rich_text(props.get("Run Policy")),
        "scope": _rich_text(props.get("Scope")),
        "owner": _rich_text(props.get("Owner")),
        "created_time": page.get("created_time", ""),
        "last_edited_time": page.get("last_edited_time", ""),
    }

def _priority_rank(p: Optional[str]) -> int:
    if not p:
        return 99
    return PRIORITY_ORDER.get(p, 99)

def _parse_iso_dt(s: str) -> datetime:
    try:
        return datetime.fromisoformat(s.replace("Z", "+00:00"))
    except Exception:
        return datetime(1970, 1, 1)

def _fmt_option(page: Dict[str, Any]) -> str:
    f = _extract_task_fields(page)
    pr = f.get("priority") or "-"
    st = f.get("status") or "-"
    dm = f.get("domain") or "-"
    ep = f.get("entry_point") or ""
    ep_short = ep if len(ep) <= 48 else ep[:48] + "…"
    return f"[{pr} | {st} | {dm}] {f.get('title') or '(untitled)'} — {ep_short}"

# -------------------------
# Fetch actionable tasks
# -------------------------

# Filter controls
domain_filter = widgets.Dropdown(
    options=[("Any", None), ("Weekly", "Weekly"), ("Daily", "Daily"), ("Targets", "Targets"), ("Papers", "Papers"), ("Events", "Events"), ("RQ", "RQ"), ("Infra", "Infra")],
    value="Weekly",
    description="Domain:",
    layout=widgets.Layout(width="420px"),
)

status_filter = widgets.SelectMultiple(
    options=["READY", "RUNNING"],
    value=("READY",),
    description="Status:",
    layout=widgets.Layout(width="420px", height="80px"),
)

refresh_btn = widgets.Button(description="Refresh", button_style="")
select_btn = widgets.Button(description="Select this task", button_style="success")

output = widgets.Output()

# Globals to be set by selection
task_page = None
TASK_PAGE_ID = None
task_fields = None

def _load_tasks():
    statuses = list(status_filter.value) or ["READY", "RUNNING"]
    dm = domain_filter.value

    pages = repos.tasks.query_tasks(
        statuses=statuses,
        domain=dm,
        page_size=50,
    )

    # Sort deterministically: priority then created_time (oldest first)
    scored: List[Tuple[int, datetime, Dict[str, Any]]] = []
    for p in pages:
        f = _extract_task_fields(p)
        pr = _priority_rank(f.get("priority"))
        ct = _parse_iso_dt(f.get("created_time") or p.get("created_time", ""))
        scored.append((pr, ct, p))
    scored.sort(key=lambda x: (x[0], x[1]))
    pages_sorted = [p for _, _, p in scored]

    return pages_sorted

def _refresh(_=None):
    with output:
        clear_output(wait=True)
        try:
            pages_sorted = _load_tasks()
            if not pages_sorted:
                print("No tasks found for the current filter.")
                task_dropdown.options = []
                return

            # Create dropdown options: (label, page_id)
            opts = [(_fmt_option(p), p["id"]) for p in pages_sorted]
            task_dropdown.options = opts
            task_dropdown.value = opts[0][1]
            print(f"Loaded {len(opts)} tasks.")
        except Exception as e:
            print("Failed to load tasks:", type(e).__name__, str(e))

task_dropdown = widgets.Dropdown(
    options=[],
    description="Task:",
    layout=widgets.Layout(width="900px"),
)

details = widgets.Output()

def _show_details(change=None):
    with details:
        clear_output(wait=True)
        if not task_dropdown.value:
            print("Select a task.")
            return
        # Retrieve selected page to show full details
        page = repos.tasks.retrieve_page(page_id=task_dropdown.value)
        f = _extract_task_fields(page)
        print("TASK_PAGE_ID:", page["id"])
        print("Title:", f["title"])
        print("Status:", f["status"])
        print("Priority:", f["priority"])
        print("Domain:", f["domain"])
        print("Entry Point:", f["entry_point"])
        print("Objective:", (f["objective"] or "")[:400])
        print("Acceptance Criteria:", (f["acceptance_criteria"] or "")[:400])
        print("Constraints:", (f["constraints"] or "")[:400])
        print("Run Policy:", (f["run_policy"] or "")[:400])

task_dropdown.observe(_show_details, names="value")

def _select(_=None):
    global task_page, TASK_PAGE_ID, task_fields
    with output:
        clear_output(wait=True)
        if not task_dropdown.value:
            print("No task selected.")
            return
        task_page = repos.tasks.retrieve_page(page_id=task_dropdown.value)
        TASK_PAGE_ID = task_page["id"]
        task_fields = _extract_task_fields(task_page)
        print("Selected Task ✅")
        print("  TASK_PAGE_ID:", TASK_PAGE_ID)
        print("  Title:", task_fields["title"])
        print("  Status:", task_fields["status"])
        print("  Priority:", task_fields["priority"])
        print("  Domain:", task_fields["domain"])
        print("  Entry Point:", task_fields["entry_point"])

refresh_btn.on_click(_refresh)
select_btn.on_click(_select)

controls = widgets.VBox([
    widgets.HBox([domain_filter, refresh_btn]),
    status_filter,
    task_dropdown,
    widgets.HBox([select_btn]),
])

display(controls, details, output)

# Initial load
_refresh()
_show_details()


Output()

Output()

In [15]:
# ============================================================
# Cell 05 — Derive execution config from selected Task
# ============================================================
# Overview:
#   Convert the selected Task into a concrete execution configuration.
#   If the target notebook does not exist, create a minimal scaffold notebook.
#
# Inputs / Outputs:
#   Inputs:
#     - task_page / TASK_PAGE_ID / task_fields (from Cell 04)
#   Outputs:
#     - target_nb_path (Path)
#     - sandbox_nb_path (Path)
#     - exec_cfg (dict)
#
# Notes:
#   - Target notebook may be auto-created if missing.
#   - This cell does NOT execute or patch anything yet.
# ============================================================

from pathlib import Path
import re
import json

# -------------------------
# Preconditions
# -------------------------
if TASK_PAGE_ID is None or task_fields is None:
    raise RuntimeError("No Task selected. Run Cell 04 and select a task first.")

entry_point = (task_fields.get("entry_point") or "").strip()
if not entry_point:
    raise RuntimeError("Selected Task is missing 'Entry Point'.")

# Normalize path
target_nb_path = Path(entry_point).expanduser()
if not target_nb_path.is_absolute():
    target_nb_path = (Path.cwd() / target_nb_path).resolve()

if target_nb_path.suffix.lower() != ".ipynb":
    raise ValueError(f"Entry Point must be a .ipynb file, got: {target_nb_path}")

# -------------------------
# Create notebook if missing
# -------------------------
if not target_nb_path.exists():
    target_nb_path.parent.mkdir(parents=True, exist_ok=True)

    minimal_nb = {
        "cells": [
            {
                "cell_type": "code",
                "execution_count": None,
                "metadata": {},
                "outputs": [],
                "source": [
                    "# Auto-generated notebook scaffold\n",
                    "# Created by 044_Notebook_builder\n",
                    "\n",
                    "print('Notebook scaffold created')\n",
                ],
            }
        ],
        "metadata": {
            "kernelspec": {
                "display_name": "Python 3",
                "language": "python",
                "name": "python3",
            },
            "language_info": {
                "name": "python",
                "version": "3.x",
            },
        },
        "nbformat": 4,
        "nbformat_minor": 5,
    }

    with open(target_nb_path, "w", encoding="utf-8") as f:
        json.dump(minimal_nb, f, indent=2)

    print(f"[info] Created new notebook scaffold at: {target_nb_path}")
else:
    print(f"[info] Using existing notebook: {target_nb_path}")

# Sandbox path
sandbox_nb_path = target_nb_path.with_name(target_nb_path.stem + "__sandbox.ipynb")

# -------------------------
# Parse run policy hints (optional)
# -------------------------
run_policy_text = (task_fields.get("run_policy") or "").strip()

def _find_int(pattern: str, text: str) -> int | None:
    m = re.search(pattern, text, flags=re.IGNORECASE)
    return int(m.group(1)) if m else None

def _find_str(pattern: str, text: str) -> str | None:
    m = re.search(pattern, text, flags=re.IGNORECASE)
    return m.group(1).strip() if m else None

# Defaults
default_prefix_timeout_sec = 180
default_full_timeout_sec = 900

prefix_timeout_sec = _find_int(r"prefix_timeout_sec\s*=\s*(\d+)", run_policy_text) or default_prefix_timeout_sec
full_timeout_sec = _find_int(r"full_timeout_sec\s*=\s*(\d+)", run_policy_text) or default_full_timeout_sec
up_to_cell_index = _find_int(r"up_to_cell_index\s*=\s*(\d+)", run_policy_text)

run_mode_hint = _find_str(r"run_mode\s*=\s*([A-Za-z_]+)", run_policy_text)
run_mode_hint = run_mode_hint.upper() if run_mode_hint else None
if run_mode_hint not in (None, "PREFIX", "FULL"):
    print(f"[warn] Unknown run_mode hint: {run_mode_hint} (ignored)")
    run_mode_hint = None

# -------------------------
# Build execution config
# -------------------------
exec_cfg = {
    "task_page_id": TASK_PAGE_ID,
    "task_title": task_fields.get("title"),
    "target_notebook_path": str(target_nb_path),
    "sandbox_notebook_path": str(sandbox_nb_path),
    "prefix_timeout_sec": int(prefix_timeout_sec),
    "full_timeout_sec": int(full_timeout_sec),
    "up_to_cell_index": up_to_cell_index,
    "run_mode_hint": run_mode_hint,
    "constraints_text": task_fields.get("constraints") or "",
    "acceptance_criteria_text": task_fields.get("acceptance_criteria") or "",
}

print("Execution config derived ✅")
print("  Task:", exec_cfg["task_title"])
print("  Target notebook:", exec_cfg["target_notebook_path"])
print("  Sandbox notebook:", exec_cfg["sandbox_notebook_path"])
print("  Prefix timeout:", exec_cfg["prefix_timeout_sec"])
print("  Full timeout:", exec_cfg["full_timeout_sec"])
print("  up_to_cell_index hint:", exec_cfg["up_to_cell_index"])
print("  run_mode hint:", exec_cfg["run_mode_hint"])


[info] Using existing notebook: /Users/yuetoya/projects/researchOS100-private/notebooks/045_weekly_discovery_expansion.ipynb
Execution config derived ✅
  Task: Weekly Discovery Expansion from Papers and Events
  Target notebook: /Users/yuetoya/projects/researchOS100-private/notebooks/045_weekly_discovery_expansion.ipynb
  Sandbox notebook: /Users/yuetoya/projects/researchOS100-private/notebooks/045_weekly_discovery_expansion__sandbox.ipynb
  Prefix timeout: 180
  Full timeout: 900
  up_to_cell_index hint: None
  run_mode hint: None


In [16]:
# ============================================================
# Cell 06 — Prepare sandbox and initialize machine state (StateStore)
# ============================================================

from pathlib import Path
import shutil
import time

from src.state.state_store import StateStore, default_state

# -------------------------
# Preconditions
# -------------------------
if "exec_cfg" not in globals():
    raise RuntimeError("exec_cfg not found. Run Cell 05 first.")

target_nb_path = Path(exec_cfg["target_notebook_path"]).resolve()
sandbox_nb_path = Path(exec_cfg["sandbox_notebook_path"]).resolve()

if not target_nb_path.exists():
    raise FileNotFoundError(f"Target notebook not found: {target_nb_path}")

# -------------------------
# Policies
# -------------------------
REFRESH_SANDBOX = True
CLEAR_QUEUE_FOR_THIS_RUN = True   # ←必要なら True に
CLEAR_SCOPE = "ALL"               # "ALL" or "TASK_PROPOSAL"
#   - "ALL": queue を全消し
#   - "TASK_PROPOSAL": task_page_id + proposal_page_id が一致する TODO/DOING/WAITING を消す（他は残す）

store = StateStore()

# Ensure sandbox dir exists
sandbox_nb_path.parent.mkdir(parents=True, exist_ok=True)

# Refresh sandbox
if (not sandbox_nb_path.exists()) or REFRESH_SANDBOX:
    shutil.copy2(target_nb_path, sandbox_nb_path)
    print(f"[info] Sandbox refreshed: {sandbox_nb_path}")
else:
    print(f"[info] Sandbox kept (no refresh): {sandbox_nb_path}")

# -------------------------
# Initialize machine state FIRST
# -------------------------
store.ensure_initialized()  # creates state/state.json if missing
print("[info] StateStore initialized:", getattr(store, "path", "state/state.json"))

# -------------------------
# Queue reset helpers
# -------------------------
def _clear_all_queue(st: dict) -> dict:
    st = dict(st or default_state())
    st["queue"] = []
    # おまけ：今回の実行印
    st["run_session"] = {"cleared_at": time.strftime("%Y-%m-%dT%H:%M:%S%z")}
    return st

def _clear_task_proposal_queue(st: dict) -> dict:
    st = dict(st or default_state())
    q = st.get("queue") or []
    if not isinstance(q, list):
        q = []

    task_page_id = str(exec_cfg.get("task_page_id") or "")
    proposal_page_id = str(globals().get("proposal_page_id") or globals().get("PROPOSAL_PAGE_ID") or "")

    if not task_page_id or not proposal_page_id:
        # task/proposal が無ければ安全側で全消しはしない
        st["run_session"] = {
            "cleared_at": time.strftime("%Y-%m-%dT%H:%M:%S%z"),
            "note": "SKIPPED clear_task_proposal_queue (missing task_page_id/proposal_page_id)",
        }
        return st

    new_q = []
    removed = 0
    for it in q:
        if not isinstance(it, dict):
            new_q.append(it)
            continue
        status = (it.get("status") or "TODO").upper()
        if status not in ("TODO", "DOING", "WAITING"):
            new_q.append(it)
            continue
        tgt = it.get("target") or {}
        if not isinstance(tgt, dict):
            new_q.append(it)
            continue
        same = (str(tgt.get("task_page_id") or "") == task_page_id) and (str(tgt.get("proposal_page_id") or "") == proposal_page_id)
        if same:
            removed += 1
            continue
        new_q.append(it)

    st["queue"] = new_q
    st["run_session"] = {
        "cleared_at": time.strftime("%Y-%m-%dT%H:%M:%S%z"),
        "scope": "TASK_PROPOSAL",
        "removed": removed,
        "task_page_id": task_page_id,
        "proposal_page_id": proposal_page_id,
    }
    return st

# -------------------------
# Optional: clear queue
# -------------------------
if CLEAR_QUEUE_FOR_THIS_RUN:
    if CLEAR_SCOPE == "TASK_PROPOSAL":
        store.update(_clear_task_proposal_queue)
        print("[info] Queue cleared for this task/proposal ✅")
    else:
        store.update(_clear_all_queue)
        print("[info] Queue cleared (ALL) ✅")
else:
    print("[info] Queue preserved (no clear)")

# Expose for downstream cells
exec_cfg["sandbox_notebook_path"] = str(sandbox_nb_path)


[info] Sandbox refreshed: /Users/yuetoya/projects/researchOS100-private/notebooks/045_weekly_discovery_expansion__sandbox.ipynb
[info] StateStore initialized: state/state.json
[info] Queue cleared (ALL) ✅


In [17]:
# ============================================================
# Cell 06.5 — Normalize sandbox notebook (ensure cell ids)
# ============================================================
# Overview:
#   Ensure each cell has an 'id' field explicitly
#   (future-proof against nbformat >=6 strict validation).
#
# Inputs / Outputs:
#   Inputs:
#     - exec_cfg["sandbox_notebook_path"]
#   Outputs:
#     - sandbox notebook rewritten with guaranteed cell ids
#
# Notes:
#   - Do NOT rely on nbformat.validator.normalize alone
#   - Explicitly assign UUIDs if missing
# ============================================================

from pathlib import Path
import nbformat
from nbformat.validator import normalize
import uuid

sandbox_nb_path = Path(exec_cfg["sandbox_notebook_path"]).resolve()

nb = nbformat.read(str(sandbox_nb_path), as_version=4)

# 👇 nbformat が期待する正規化（MissingID 対策）
normalize(nb)

changed = False
for cell in nb.cells:
    if not cell.get("id"):
        cell["id"] = uuid.uuid4().hex
        changed = True

nbformat.write(nb, str(sandbox_nb_path))

print(
    "[info] Normalized sandbox notebook (cell ids ensured):",
    sandbox_nb_path,
    "(updated)" if changed else "(already ok)",
)


[info] Normalized sandbox notebook (cell ids ensured): /Users/yuetoya/projects/researchOS100-private/notebooks/045_weekly_discovery_expansion__sandbox.ipynb (already ok)


In [18]:
# ============================================================
# Cell 07 — Create or resume a working Proposal chain for this Task
# ============================================================
# Overview:
#   Ensure there is exactly one "current" working Proposal (ChangeSet) for the
#   selected Task. This creates a durable ChangeSet ledger in PROPOSALS_DB.
#
# Behavior:
#   - If the Task already has a non-terminal Proposal, resume it.
#   - Else create a new initial Proposal (Status=DRAFT) tied to the Task.
#   - Always link the chosen Proposal to the Task and set it as Latest Proposal.
#
# Notes:
#   - Idempotent-ish: prefers reusing an existing non-terminal proposal.
#   - Robust to Notion property type differences (select / rich_text / number).
# ============================================================

from typing import Optional, Dict, Any, List, Tuple
import ipywidgets as widgets
from IPython.display import display, clear_output

# -------------------------
# Preconditions
# -------------------------
if TASK_PAGE_ID is None or task_page is None or task_fields is None:
    raise RuntimeError("No Task selected. Run Cell 04 first.")
if "exec_cfg" not in globals():
    raise RuntimeError("exec_cfg missing. Run Cell 05 first.")
if "repos" not in globals():
    raise RuntimeError("repos missing. Run Cell 03 first.")

# -------------------------
# Proposal status sets (adjust if your schema uses different labels)
# -------------------------
TERMINAL_STATUSES = {"VERIFIED", "FAILED", "ARCHIVED"}

# -------------------------
# Helpers (Notion property readers; robust to schema variations)
# -------------------------
def _select_name(prop: Dict[str, Any]) -> Optional[str]:
    if not prop or prop.get("type") != "select":
        return None
    v = prop.get("select")
    return v.get("name") if v else None

def _rich_text(prop: Dict[str, Any]) -> str:
    if not prop:
        return ""
    t = prop.get("type")
    if t == "rich_text":
        parts = prop.get("rich_text") or []
        return "".join(p.get("plain_text", "") for p in parts)
    if t == "title":
        parts = prop.get("title") or []
        return "".join(p.get("plain_text", "") for p in parts)
    return ""

def _number(prop: Dict[str, Any]) -> Optional[float]:
    if not prop or prop.get("type") != "number":
        return None
    return prop.get("number")

def _relation_ids(prop: Dict[str, Any]) -> List[str]:
    if not prop or prop.get("type") != "relation":
        return []
    rel = prop.get("relation") or []
    return [r.get("id") for r in rel if r.get("id")]

def _prop(props: Dict[str, Any], *names: str) -> Optional[Dict[str, Any]]:
    """Return first matching property dict by name (best-effort)."""
    for n in names:
        if n in props and isinstance(props.get(n), dict):
            return props.get(n)
    return None

def _cell_index_value(prop: Optional[Dict[str, Any]]) -> Optional[int]:
    """Cell Index may be number or rich_text/title. Normalize to int if possible."""
    if not prop:
        return None
    n = _number(prop)
    if n is not None:
        try:
            return int(n)
        except Exception:
            return None
    s = _rich_text(prop).strip()
    if not s:
        return None
    try:
        return int(float(s))
    except Exception:
        return None

def _extract_proposal_fields(page: Dict[str, Any]) -> Dict[str, Any]:
    props = page.get("properties") or {}

    title_prop = _prop(props, "Title", "Name")
    status_prop = _prop(props, "Status")
    task_prop = _prop(props, "Task", "Tasks")
    nbpath_prop = _prop(props, "Notebook Path", "NotebookPath", "Notebook")
    cellidx_prop = _prop(props, "Cell Index", "CellIndex")
    intent_prop = _prop(props, "Intent")
    acceptance_prop = _prop(props, "Acceptance", "Acceptance Criteria")
    risk_prop = _prop(props, "Risk")
    next_action_prop = _prop(props, "Next Action", "NextAction")
    failure_reason_prop = _prop(props, "Failure Reason", "FailureReason")
    last_run_prop = _prop(props, "Last Run", "LastRun")

    return {
        "title": _rich_text(title_prop),
        "status": _select_name(status_prop),
        "task_rel": _relation_ids(task_prop),
        "notebook_path": _rich_text(nbpath_prop),
        "cell_index": _cell_index_value(cellidx_prop),
        "intent": _rich_text(intent_prop),
        "acceptance": _rich_text(acceptance_prop),
        "risk": _select_name(risk_prop),
        "next_action": _rich_text(next_action_prop),
        "failure_reason": _rich_text(failure_reason_prop),
        "last_run_rel": _relation_ids(last_run_prop),
    }

def _task_proposal_ids(task_page: Dict[str, Any]) -> List[str]:
    props = task_page.get("properties") or {}
    # Prefer "Proposals" relation; fallback to "Latest Proposal"
    ids = _relation_ids(_prop(props, "Proposals", "Proposal", "All Proposals"))
    if ids:
        return ids
    return _relation_ids(_prop(props, "Latest Proposal", "LatestProposal"))

def _is_resumable_status(status: Optional[str]) -> bool:
    """Treat anything not terminal as resumable (more robust than ACTIVE whitelist)."""
    if not status:
        return True  # unknown -> treat as resumable candidate
    return str(status).upper() not in TERMINAL_STATUSES

def _safe_retrieve_proposal(pid: str) -> Optional[Dict[str, Any]]:
    try:
        return repos.proposals.retrieve_page(page_id=pid)
    except Exception:
        return None

# -------------------------
# Load existing proposals linked from the Task
# -------------------------
linked_ids = _task_proposal_ids(task_page)

linked_pages: List[Dict[str, Any]] = []
for pid in linked_ids[:50]:
    p = _safe_retrieve_proposal(pid)
    if p:
        linked_pages.append(p)

# resumable candidates (non-terminal)
resumable: List[Dict[str, Any]] = []
for p in linked_pages:
    st = _select_name((p.get("properties") or {}).get("Status"))
    if _is_resumable_status(st):
        resumable.append(p)

# Outputs
proposal_page: Optional[Dict[str, Any]] = None
PROPOSAL_PAGE_ID: Optional[str] = None
proposal_fields: Optional[Dict[str, Any]] = None

out = widgets.Output()

def _set_current_proposal(pid: str) -> Tuple[Dict[str, Any], Dict[str, Any]]:
    """Retrieve + set latest link for the Task, return (proposal_page, proposal_fields)."""
    p = repos.proposals.retrieve_page(page_id=pid)
    repos.tasks.link_proposal(task_page_id=TASK_PAGE_ID, proposal_page_id=pid, set_latest=True)
    f = _extract_proposal_fields(p)
    return p, f

def _choose_or_create():
    global proposal_page, PROPOSAL_PAGE_ID, proposal_fields, task_page

    with out:
        clear_output(wait=True)

        if resumable:
            # If multiple resumables exist, warn; we will pick one deterministically unless user selects.
            if len(resumable) == 1:
                proposal_page = resumable[0]
                PROPOSAL_PAGE_ID = proposal_page["id"]
                proposal_fields = _extract_proposal_fields(proposal_page)

                # Ensure linkage + set latest (idempotent)
                repos.tasks.link_proposal(task_page_id=TASK_PAGE_ID, proposal_page_id=PROPOSAL_PAGE_ID, set_latest=True)

                print("[info] Resuming existing Proposal:", PROPOSAL_PAGE_ID)
                print("  Title:", proposal_fields.get("title"))
                print("  Status:", proposal_fields.get("status"))
                return

            print(f"[warn] Multiple resumable proposals found ({len(resumable)}).")
            print("       Choose one to continue. (We will set it as Latest Proposal.)")

            options = []
            for p in resumable:
                f = _extract_proposal_fields(p)
                label = f"[{f.get('status') or 'UNKNOWN'}] {f.get('title') or '(no title)'} — id={p.get('id')}"
                options.append((label, p["id"]))

            dd = widgets.Dropdown(options=options, description="Resume:", layout=widgets.Layout(width="950px"))
            btn = widgets.Button(description="Use selected", button_style="success")

            def _use_selected(_=None):
                global proposal_page, PROPOSAL_PAGE_ID, proposal_fields, task_page
                pid = dd.value
                # Keep output clean after selection
                clear_output(wait=True)

                proposal_page, proposal_fields = _set_current_proposal(pid)
                PROPOSAL_PAGE_ID = proposal_page["id"]

                # Refresh task_page (so downstream sees updated relations)
                task_page = repos.tasks.retrieve_page(page_id=TASK_PAGE_ID)

                print("[info] Resuming selected Proposal:", PROPOSAL_PAGE_ID)
                print("  Title:", proposal_fields.get("title"))
                print("  Status:", proposal_fields.get("status"))

            btn.on_click(_use_selected)
            display(widgets.VBox([dd, btn]))
            return

        # No resumable proposals → create new initial DRAFT proposal
        title = f"INIT: {task_fields.get('title') or 'Notebook build'}"
        proposal_page = repos.proposals.create_changeset(
            title=title,
            task_page_id=TASK_PAGE_ID,
            status="DRAFT",
            notebook_path=exec_cfg.get("sandbox_notebook_path") or "",
            cell_index=0,
            intent="Initialize ChangeSet ledger for this Task (builder session start).",
            acceptance="A first patch + verification run can be executed against the sandbox notebook.",
            risk="LOW",
        )
        PROPOSAL_PAGE_ID = proposal_page["id"]

        # Link proposal to Task and set as latest
        repos.tasks.link_proposal(task_page_id=TASK_PAGE_ID, proposal_page_id=PROPOSAL_PAGE_ID, set_latest=True)

        # Refresh task_page
        task_page = repos.tasks.retrieve_page(page_id=TASK_PAGE_ID)

        proposal_fields = _extract_proposal_fields(proposal_page)
        print("[info] Created new Proposal:", PROPOSAL_PAGE_ID)
        print("  Title:", proposal_fields.get("title"))
        print("  Status:", proposal_fields.get("status"))
        print("  Notebook Path:", proposal_fields.get("notebook_path"))

_choose_or_create()
display(out)
proposal_page_id = PROPOSAL_PAGE_ID

Output()

In [19]:
import sys, importlib
import src.notion.client as nc
import src.nb.patcher as patcher
import src.llm.claude_client as claude_client
import src.orchestrator.steps_llm as steps_llm
import src.orchestrator.one_loop as one_loop
import src.nb.scaffold_headers as scaffold_headers
import src.exec.nb_runner as nb_runner
import src.state.state_store as state_store
import src.verify.error_parser as error_parser

importlib.reload(nc)
importlib.reload(patcher)
importlib.reload(claude_client)
importlib.reload(steps_llm)
importlib.reload(scaffold_headers)
importlib.reload(one_loop)
importlib.reload(nb_runner)
importlib.reload(state_store)
importlib.reload(error_parser)

print("reloaded module:", nc.__file__)
print("reloaded module:", patcher.__file__)
print("reloaded module:", claude_client.__file__)
print("reloaded module:", steps_llm.__file__)
print("reloaded module:", one_loop.__file__)
print("reloaded module:", scaffold_headers.__file__)
print("reloaded module:", nb_runner.__file__)
print("reloaded module:", state_store.__file__)
print("reloaded module:", error_parser.__file__)


reloaded module: /Users/yuetoya/projects/researchOS100-private/src/notion/client.py
reloaded module: /Users/yuetoya/projects/researchOS100-private/src/nb/patcher.py
reloaded module: /Users/yuetoya/projects/researchOS100-private/src/llm/claude_client.py
reloaded module: /Users/yuetoya/projects/researchOS100-private/src/orchestrator/steps_llm.py
reloaded module: /Users/yuetoya/projects/researchOS100-private/src/orchestrator/one_loop.py
reloaded module: /Users/yuetoya/projects/researchOS100-private/src/nb/scaffold_headers.py
reloaded module: /Users/yuetoya/projects/researchOS100-private/src/exec/nb_runner.py
reloaded module: /Users/yuetoya/projects/researchOS100-private/src/state/state_store.py
reloaded module: /Users/yuetoya/projects/researchOS100-private/src/verify/error_parser.py


In [20]:
# ============================================================
# Cell 07.9 — Policy (task constraints)
# ============================================================

policy = {
    # Cell00に「Required env vars」として列挙される
    "required_env": [
        "NOTION_LIT_DB_ID",
        "NOTION_EVENTS_DB_ID",
        "NOTION_MONITORING_TARGETS_DB_ID",
        "NOTION_RQ_DB_ID",
        "NOTION_WEEKLY_TARGET_UPDATE_DB_ID",
    ],

    # Cell00に「Data sources (read scope)」として表示される（表示用途）
    "data_sources": {
        "papers_db": "NOTION_LIT_DB_ID",
        "events_db": "NOTION_EVENTS_DB_ID",
        "monitoring_targets_db": "NOTION_MONITORING_TARGETS_DB_ID",
        "rq_db": "NOTION_RQ_DB_ID",
    },

    # Cell00に「Writes are restricted to」と「Write prohibitions」として表示される
    "writes_allowed": ["NOTION_WEEKLY_TARGET_UPDATE_DB_ID"],
    "writes_forbidden": ["NOTION_MONITORING_TARGETS_DB_ID"],

    # digestにも効く＆Cell00の「Execution & Safety Constraints」に表示される
    "safety": {
        "truncate_rich_text_max_chars": 1800,
        "bounded_retries": True,
        "append_only_writes": True,
        "scope_window_days": 7,
        "output_mode": "candidates_only",
    },

    # これは scaffold_headers の policy["structure"] と衝突しないよう別名に
    "scaffold_flags": {"cell00_overview": True, "cell_headers": True},
}

print("policy loaded ✅")


policy loaded ✅


In [21]:
# ============================================================
# Cell 08 — Builder: enqueue SCAFFOLD_HEADERS (00/01/02) then VERIFY (PREFIX up to 02)
#        - Do NOT enqueue LLM_PLAN here.
#        - Plan for Cell03+ is triggered ONLY after verify pass via a state flag.
# ============================================================

import time
import hashlib
import json
from typing import Any, Dict, List

from src.state.state_store import new_task_item, enqueue

# -------------------------
# Resolve globals
# -------------------------
task_page_id = globals().get("task_page_id") or globals().get("TASK_PAGE_ID")
assert task_page_id, "task_page_id/TASK_PAGE_ID is missing (select a task in Cell 04/05)."
assert "proposal_page_id" in globals(), "proposal_page_id is missing (create/resume proposal in Cell 07)."
assert "exec_cfg" in globals() and exec_cfg.get("sandbox_notebook_path"), "exec_cfg['sandbox_notebook_path'] missing."
assert "store" in globals() and "repos" in globals(), "store/repos missing (Cell 03)."
assert "task_fields" in globals() and isinstance(task_fields, dict), "task_fields missing (Cell 04/05)."

proposal_page_id = globals()["proposal_page_id"]
sandbox_nb_path = exec_cfg["sandbox_notebook_path"]

# -------------------------
# Helpers
# -------------------------
def _stable_digest(obj: Any) -> str:
    s = json.dumps(obj, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(s.encode("utf-8")).hexdigest()[:16]

def _now_iso() -> str:
    return time.strftime("%Y-%m-%dT%H:%M:%S%z")

def _cancel_todo_for_same_task_proposal(
    st: Dict[str, Any],
    *,
    task_page_id: str,
    proposal_page_id: str,
    types: List[str],
) -> int:
    """Mark TODO items as CANCELLED for same task/proposal and given types."""
    types_u = {t.upper() for t in types}
    q = st.get("queue") or []
    if not isinstance(q, list):
        return 0

    cancelled = 0
    new_q = []
    for it in q:
        if not isinstance(it, dict):
            new_q.append(it)
            continue

        if (it.get("status") or "TODO") != "TODO":
            new_q.append(it)
            continue

        t = (it.get("type") or "").upper()
        if t not in types_u:
            new_q.append(it)
            continue

        tgt = it.get("target") or {}
        if not isinstance(tgt, dict):
            new_q.append(it)
            continue

        same = (str(tgt.get("task_page_id") or "") == str(task_page_id)) and (
            str(tgt.get("proposal_page_id") or "") == str(proposal_page_id)
        )
        if not same:
            new_q.append(it)
            continue

        it2 = dict(it)
        it2["status"] = "CANCELLED"
        it2["last_error"] = "Superseded by builder enqueue (bootstrap 00/01/02)"
        it2["updated_at"] = _now_iso()
        new_q.append(it2)
        cancelled += 1

    st["queue"] = new_q
    return cancelled

def _state_patch(store, patch: Dict[str, Any]) -> None:
    def _fn(st: Dict[str, Any]) -> Dict[str, Any]:
        if not isinstance(st, dict):
            st = {}
        for k, v in (patch or {}).items():
            st[k] = v
        return st
    store.update(_fn)

# -------------------------
# Policy (prefer globals; else fallback)
# NOTE: scaffold_headers.py 側が writes_allowed/forbidden を見る想定なら最低限ここは維持
# -------------------------
if "policy" in globals() and isinstance(policy, dict):
    _policy = policy
else:
    _policy = {
        "required_env": [
            "NOTION_LIT_DB_ID",
            "NOTION_EVENTS_DB_ID",
            "NOTION_MONITORING_TARGETS_DB_ID",
            "NOTION_RQ_DB_ID",
            "NOTION_WEEKLY_TARGET_UPDATE_DB_ID",
        ],
        "data_sources": {
            "papers_db": "NOTION_LIT_DB_ID",
            "events_db": "NOTION_EVENTS_DB_ID",
            "monitoring_targets_db": "NOTION_MONITORING_TARGETS_DB_ID",
            "rq_db": "NOTION_RQ_DB_ID",
        },
        "writes_allowed": ["NOTION_WEEKLY_TARGET_UPDATE_DB_ID"],
        "writes_forbidden": [
            "NOTION_MONITORING_TARGETS_DB_ID",
            "NOTION_EVENTS_DB_ID",
            "NOTION_LIT_DB_ID",
            "NOTION_RQ_DB_ID",
        ],
        "safety": {
            "truncate_rich_text_max_chars": 1000,
            "append_only_writes": True,
            "scope_window_days": 7,
        },
        "scaffold_flags": {"cell00_overview": True, "cell_headers": True},
    }

policy = _policy

# -------------------------
# Debounce signature (scaffold+verify bootstrap only)
# -------------------------
sig_obj = {
    "task_page_id": str(task_page_id),
    "proposal_page_id": str(proposal_page_id),
    "notebook_path": str(sandbox_nb_path),
    "policy": policy,
    "bootstrap": {"cells": [0, 1, 2], "verify_prefix_up_to": 2},
    "task_fields_subset": {
        "title": task_fields.get("title"),
        "entry_point": task_fields.get("entry_point"),
        "objective": task_fields.get("objective"),
        "acceptance_criteria": task_fields.get("acceptance_criteria"),
        "constraints": task_fields.get("constraints"),
        "run_policy": task_fields.get("run_policy"),
        "scope": task_fields.get("scope"),
    },
}
digest = _stable_digest(sig_obj)

# -------------------------
# Debounce + cancel duplicates (single update)
# -------------------------
did_skip = {"skipped": False, "cancelled": 0}

def _update(st: Dict[str, Any]) -> Dict[str, Any]:
    if not isinstance(st, dict):
        st = {}

    deb = st.setdefault("builder_debounce", {})
    last_digest = deb.get("digest")

    # check if same bootstrap already queued for same task/proposal
    def _has_same_todo() -> bool:
        q = st.get("queue") or []
        if not isinstance(q, list):
            return False
        for it in q:
            if not isinstance(it, dict):
                continue
            if (it.get("status") or "TODO") != "TODO":
                continue
            if (it.get("type") or "").upper() not in ("SCAFFOLD_HEADERS", "VERIFY_NOTEBOOK"):
                continue
            tgt = it.get("target") or {}
            if str(tgt.get("task_page_id") or "") != str(task_page_id):
                continue
            if str(tgt.get("proposal_page_id") or "") != str(proposal_page_id):
                continue
            # verifyは up_to=2 のものだけに限定（別verifyと混ぜない）
            if (it.get("type") or "").upper() == "VERIFY_NOTEBOOK":
                if str((tgt.get("run_mode") or "")).upper() != "PREFIX":
                    continue
                if int(tgt.get("up_to_cell_index") or -1) != 2:
                    continue
            return True
        return False

    if last_digest == digest and _has_same_todo():
        deb["digest"] = last_digest
        deb["ts"] = deb.get("ts")
        deb["skipped"] = True
        did_skip["skipped"] = True
        return st

    # mark debounce updated
    st["builder_debounce"] = {"digest": digest, "ts": _now_iso(), "skipped": False}
    did_skip["skipped"] = False

    # cancel stale TODOs for same task/proposal
    did_skip["cancelled"] = _cancel_todo_for_same_task_proposal(
        st,
        task_page_id=str(task_page_id),
        proposal_page_id=str(proposal_page_id),
        types=["SCAFFOLD_HEADERS", "VERIFY_NOTEBOOK"],
    )

    # set post-bootstrap plan trigger (consumed by one_loop after verify pass)
    pb = st.setdefault("post_bootstrap_plan", {})
    pb[str(proposal_page_id)] = {
        "pending": True,
        "start_cell_index": 3,
        "mode": "ONE_BY_ONE",
        "notebook_path": str(sandbox_nb_path),
        "task_page_id": str(task_page_id),
        "proposal_page_id": str(proposal_page_id),
        "updated_at": _now_iso(),
    }
    st["post_bootstrap_plan"] = pb

    return st

store.update(_update)

if not did_skip["skipped"]:
    # --------------------------------------------------------
    # 1) Enqueue SCAFFOLD_HEADERS
    #    - scaffold_headers.py 側が Cell01/02 を "FORCE" で置く想定
    #    - structure は空でも良い（Cell03+はここでは触らない）
    # --------------------------------------------------------
    scaffold_target = {
        "task_page_id": str(task_page_id),
        "proposal_page_id": str(proposal_page_id),
        "notebook_path": str(sandbox_nb_path),
        "task_fields": dict(task_fields),
        "policy": dict(policy),
        "structure": [],               # ✅ Cell03+は作らない/触らない
        "cleanup_queue": True,
        "preserve_existing": True,
    }
    enqueue(
        store,
        new_task_item(
            type="SCAFFOLD_HEADERS",
            intent="Bootstrap scaffold: create/replace Cell00/01/02 only",
            assignee="SYSTEM",
            priority=99,
            target=scaffold_target,
        ),
    )

    # --------------------------------------------------------
    # 2) Enqueue VERIFY_NOTEBOOK (PREFIX up to Cell02)
    # --------------------------------------------------------
    verify_target = {
        "task_page_id": str(task_page_id),
        "proposal_page_id": str(proposal_page_id),
        "notebook_path": str(sandbox_nb_path),
        "run_mode": "PREFIX",
        "up_to_cell_index": 2,
        "timeout_sec": 300,
        "quality_gates": {
            "ruff": {"enabled": False, "args": ["check", "."], "timeout_sec": 300, "cwd": "."},
            "pytest": {"enabled": False, "args": ["-q"], "timeout_sec": 900, "cwd": "."},
        },
        # ✅ verify失敗時の自動replanはone_loop側で制御
        "auto_replan_on_fail": True,
        "replan_debounce_sec": 120,
    }
    enqueue(
        store,
        new_task_item(
            type="VERIFY_NOTEBOOK",
            intent="Bootstrap verify: PREFIX up to Cell02",
            assignee="VERIFIER",
            priority=98,
            target=verify_target,
        ),
    )

st_now = store.load() if hasattr(store, "load") else {}
q = (st_now or {}).get("queue") or []
print("Bootstrap builder enqueue ✅")
print("digest:", digest)
print("skipped:", bool(did_skip["skipped"]))
print("cancelled:", int(did_skip["cancelled"]))
print("queue_types:", [(it.get("type"), it.get("status")) for it in q if isinstance(it, dict)])


Bootstrap builder enqueue ✅
digest: f5dc8b83cc02b6ae
skipped: False
cancelled: 0
queue_types: [('SCAFFOLD_HEADERS', 'TODO'), ('VERIFY_NOTEBOOK', 'TODO')]


In [ ]:
# ============================================================
# Cell 09 — Execute one-loop until queue idle (LLM-aware) [safer]
# ============================================================

import time
import traceback
from src.orchestrator.one_loop import run_one_step
from src.orchestrator.steps_llm import handle_llm_step
from src.state.state_store import enqueue, new_task_item

MAX_STEPS = 200
SLEEP_SEC = 0.0
VERBOSE = True

def _st_read(store):
    if hasattr(store, "load"):
        st = store.load()
        # Some StateStore variants may return JSON string; normalize to dict.
        if isinstance(st, dict):
            return st
        if isinstance(st, str):
            try:
                import json
                obj = json.loads(st)
                return obj if isinstance(obj, dict) else {}
            except Exception:
                return {}
        return {}
    from src.state.state_store import read_state
    st = read_state(store)
    if isinstance(st, dict):
        return st
    if isinstance(st, str):
        try:
            import json
            obj = json.loads(st)
            return obj if isinstance(obj, dict) else {}
        except Exception:
            return {}
    return {}

def _find_next_todo(queue):
    for it in (queue or []):
        if (it.get("status") or "TODO") == "TODO":
            return it
    return None

def _queue_fingerprint(queue, n=8):
    """
    small stable snapshot to detect 'no progress' loops
    """
    out = []
    for it in (queue or [])[:n]:
        if not isinstance(it, dict):
            continue
        out.append(
            (
                it.get("task_item_id"),
                (it.get("type") or "").upper(),
                it.get("status"),
                it.get("attempts"),
                it.get("updated_at"),
            )
        )
    return tuple(out)

print("[start] Running loop…")

# ------------------------------------------------------------
# ✅ PRE-SEED (DIRECT RUN): Tasks -> LLM -> state.plan_seed_structure
# ------------------------------------------------------------
st = _st_read(store)
queue = st.get("queue") or []

seeded = st.get("plan_seed_structure")
has_seed = isinstance(seeded, list) and len(seeded) > 0

if not has_seed:
    any_item = next((it for it in queue if isinstance(it, dict) and isinstance(it.get("target"), dict)), None)
    tgt = dict((any_item or {}).get("target") or {})

    if tgt.get("task_page_id") and tgt.get("proposal_page_id") and tgt.get("notebook_path"):
        # ✅ seed-only を “直実行” して state に入れる
        _ = handle_llm_step(
            step_type="LLM_PLAN",
            store=store,
            repos=repos,
            claude=claude,
            task_item_id=str((any_item or {}).get("task_item_id") or "preloop_seed"),
            target={
                "task_page_id": tgt["task_page_id"],
                "proposal_page_id": tgt["proposal_page_id"],
                "notebook_path": tgt["notebook_path"],
                "llm": tgt.get("llm") or {},
                "policy": tgt.get("policy") or {},
                "hint": {"phase": "STRUCTURE_SEED"},
                "structure_seed_only": True,
            },
        )

# refresh after pre-seed
st = _st_read(store)
queue = st.get("queue") or []

step = 0
prev_fp = None
no_progress = 0

while step < MAX_STEPS:
    step += 1

    st = _st_read(store)
    queue = st.get("queue") or []

    # ✅ Hard-guard: if seed is missing, do NOT proceed
    seeded = st.get("plan_seed_structure")
    has_seed = isinstance(seeded, list) and len(seeded) > 0
    if not has_seed:
        print("[stop] plan_seed_structure is missing; seed-only failed or task context missing.")
        break

    fp = _queue_fingerprint(queue)
    if fp == prev_fp:
        no_progress += 1
    else:
        no_progress = 0
    prev_fp = fp

    if no_progress >= 20:
        print("[stop] No progress detected (queue fingerprint unchanged).")
        break

    todo = _find_next_todo(queue)
    if todo is None:
        print("[stop] Queue idle (no TODO items).")
        break

    peek_type = (todo.get("type") or "").upper()
    peek_id = todo.get("task_item_id")

    try:
        if peek_type in ("LLM_PLAN", "LLM_IMPLEMENT"):
            res = handle_llm_step(
                step_type=peek_type,
                store=store,
                repos=repos,
                claude=claude,
                task_item_id=peek_id,
                target=dict(todo.get("target") or {}),
            )
            ok = bool(res.get("ok"))
            msg = str(res.get("message") or "")
            out_type = peek_type
            out_id = peek_id
        else:
            r = run_one_step(store=store, repos=repos)
            ok = bool(r.ok)
            msg = str(r.message or "")
            out_type = (r.step_type or peek_type)
            out_id = r.task_item_id or peek_id

        if VERBOSE:
            status = "ok=True" if ok else "ok=False"
            print(f"[{step:03d}] type={out_type} id={out_id} {status} msg={msg}")

    except Exception as e:
        print(f"[{step:03d}] EXCEPTION {type(e).__name__}: {e}")
        traceback.print_exc()
        break

print("Loop finished ✅")
print("Total steps executed:", step)



[start] Running loop…

[STRUCTURE][DEBOUNCED] extracted from Cell00:
[
  {
    "cell_index": 1,
    "title": "Setup & Configuration",
    "overview": "Initialize environment, import required libraries, and configure system parameters",
    "io": "Input: None. Output: Configured environment ready for data processing",
    "notes": "Essential setup cell - must run first"
  },
  {
    "cell_index": 2,
    "title": "Schema Definition",
    "overview": "Define and validate the data schema structure",
    "io": "Input: Schema requirements. Output: Validated schema object",
    "notes": "Fixed schema truth cell - do not modify"
  },
  {
    "cell_index": 3,
    "title": "Data Ingestion",
    "overview": "Load and parse input data from specified sources",
    "io": "Input: Data source paths/connections. Output: Raw data objects",
    "notes": "Handles multiple data formats and sources"
  },
  {
    "cell_index": 4,
    "title": "Data Validation & Cleaning",
    "overview": "Validate data again